In [1]:
# Lab 3 - GridWorld - Policy Evaluation and Value Iteration
import random
random.seed(1)

In [2]:
# grid settings
rows = 5
cols = 5
start = (0, 0)
goal = (4, 4)
actions = ["UP", "DOWN", "LEFT", "RIGHT"]
gamma = 0.9
threshold = 0.0001

all_states = [(r, c) for r in range(rows) for c in range(cols)]
print("total states:", len(all_states))

total states: 25


In [3]:
# step function, takes state and action, returns next state, reward, done
def step(state, action):
    r, c = state
    if action == "UP":
        r = max(r - 1, 0)
    elif action == "DOWN":
        r = min(r + 1, rows - 1)
    elif action == "LEFT":
        c = max(c - 1, 0)
    elif action == "RIGHT":
        c = min(c + 1, cols - 1)

    next_state = (r, c)

    if next_state == goal:
        reward = 10
    else:
        reward = -1

    done = next_state == goal
    return next_state, reward, done

print(step((0,0), "DOWN"))
print(step((4,4), "RIGHT"))
print(step((0,0), "UP"))

((1, 0), -1, False)
((4, 4), 10, True)
((0, 0), -1, False)


In [4]:
# fixed policy to evaluate - go down till last row then go right
def fixed_policy(state):
    r, c = state
    if r < rows - 1:
        return "DOWN"
    else:
        return "RIGHT"

print(fixed_policy((0,0)))
print(fixed_policy((4,2)))

DOWN
RIGHT


In [5]:
# policy evaluation
V = {s: 0 for s in all_states}
iteration = 0
delta_list = []

while True:
    delta = 0
    for s in all_states:
        if s == goal:
            continue
        a = fixed_policy(s)
        next_s, r, done = step(s, a)
        new_v = r + gamma * V[next_s]
        delta = max(delta, abs(new_v - V[s]))
        V[s] = new_v

    iteration += 1
    delta_list.append(delta)
    print("iteration", iteration, "max change =", round(delta, 4))

    if delta < threshold:
        print("converged")
        break

iteration 1 max change = 10.0
iteration 2 max change = 9.0
iteration 3 max change = 8.1
iteration 4 max change = 7.29
iteration 5 max change = 6.561
iteration 6 max change = 5.9049
iteration 7 max change = 5.3144
iteration 8 max change = 4.783
iteration 9 max change = 0
converged


In [6]:
# print value function after policy evaluation
print("state values after policy evaluation:")
for r in range(rows):
    row_vals = [round(V[(r, c)], 2) for c in range(cols)]
    print(row_vals)

state values after policy evaluation:
[-0.43, 0.63, 1.81, 3.12, 4.58]
[0.63, 1.81, 3.12, 4.58, 6.2]
[1.81, 3.12, 4.58, 6.2, 8.0]
[3.12, 4.58, 6.2, 8.0, 10.0]
[4.58, 6.2, 8.0, 10.0, 0]


In [7]:
# value iteration
V2 = {s: 0 for s in all_states}
iteration2 = 0

while True:
    delta = 0
    for s in all_states:
        if s == goal:
            continue
        best_value = -9999
        for a in actions:
            next_s, r, done = step(s, a)
            val = r + gamma * V2[next_s]
            if val > best_value:
                best_value = val
        delta = max(delta, abs(best_value - V2[s]))
        V2[s] = best_value

    iteration2 += 1
    print("iteration", iteration2, "max change =", round(delta, 4))

    if delta < threshold:
        print("converged")
        break

iteration 1 max change = 10.0
iteration 2 max change = 9.0
iteration 3 max change = 8.1
iteration 4 max change = 7.29
iteration 5 max change = 6.561
iteration 6 max change = 5.9049
iteration 7 max change = 5.3144
iteration 8 max change = 4.783
iteration 9 max change = 0
converged


In [8]:
# print optimal value function
print("optimal state values:")
for r in range(rows):
    row_vals = [round(V2[(r, c)], 2) for c in range(cols)]
    print(row_vals)

optimal state values:
[-0.43, 0.63, 1.81, 3.12, 4.58]
[0.63, 1.81, 3.12, 4.58, 6.2]
[1.81, 3.12, 4.58, 6.2, 8.0]
[3.12, 4.58, 6.2, 8.0, 10.0]
[4.58, 6.2, 8.0, 10.0, 0]


In [9]:
# extract optimal policy from V2
optimal_policy = {}
for s in all_states:
    if s == goal:
        optimal_policy[s] = "GOAL"
        continue
    best_a = None
    best_value = -9999
    for a in actions:
        next_s, r, done = step(s, a)
        val = r + gamma * V2[next_s]
        if val > best_value:
            best_value = val
            best_a = a
    optimal_policy[s] = best_a

print(optimal_policy)

{(0, 0): 'DOWN', (0, 1): 'DOWN', (0, 2): 'DOWN', (0, 3): 'DOWN', (0, 4): 'DOWN', (1, 0): 'DOWN', (1, 1): 'DOWN', (1, 2): 'DOWN', (1, 3): 'DOWN', (1, 4): 'DOWN', (2, 0): 'DOWN', (2, 1): 'DOWN', (2, 2): 'DOWN', (2, 3): 'DOWN', (2, 4): 'DOWN', (3, 0): 'DOWN', (3, 1): 'DOWN', (3, 2): 'DOWN', (3, 3): 'DOWN', (3, 4): 'DOWN', (4, 0): 'RIGHT', (4, 1): 'RIGHT', (4, 2): 'RIGHT', (4, 3): 'RIGHT', (4, 4): 'GOAL'}


In [10]:
# show optimal policy as arrows
arrows = {"UP": "up", "DOWN": "down", "LEFT": "left", "RIGHT": "right"}

print("optimal policy grid:")
for r in range(rows):
    row_symbols = []
    for c in range(cols):
        a = optimal_policy[(r, c)]
        if a == "GOAL":
            row_symbols.append("GOAL")
        else:
            row_symbols.append(arrows[a])
    print(row_symbols)

optimal policy grid:
['down', 'down', 'down', 'down', 'down']
['down', 'down', 'down', 'down', 'down']
['down', 'down', 'down', 'down', 'down']
['down', 'down', 'down', 'down', 'down']
['right', 'right', 'right', 'right', 'GOAL']


In [11]:
# function to run one episode given a policy
def run_episode(policy_func, max_steps=50):
    state = start
    path = [state]
    total_reward = 0
    steps = 0

    for i in range(max_steps):
        if state == goal:
            break
        a = policy_func(state)
        next_s, r, done = step(state, a)
        total_reward += r
        path.append(next_s)
        state = next_s
        steps += 1
        if done:
            break

    reached = (state == goal)
    return path, steps, total_reward, reached

In [12]:
# random policy
def random_policy(state):
    return random.choice(actions)

path, steps, total_reward, reached = run_episode(random_policy)
print("random policy")
print("path:", path)
print("steps:", steps)
print("total reward:", total_reward)
print("goal reached:", reached)

random policy
path: [(0, 0), (1, 0), (0, 0), (0, 0), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (0, 4), (0, 4), (0, 4), (0, 4), (0, 4), (0, 4), (0, 4), (0, 3), (1, 3), (0, 3), (0, 2), (0, 2), (0, 2), (0, 2), (0, 2), (0, 3), (1, 3), (1, 4), (0, 4), (1, 4), (1, 4), (1, 4), (2, 4), (2, 3), (3, 3), (4, 3), (4, 4)]
steps: 36
total reward: -25
goal reached: True


In [13]:
# evaluated policy
path, steps, total_reward, reached = run_episode(fixed_policy)
print("evaluated policy")
print("path:", path)
print("steps:", steps)
print("total reward:", total_reward)
print("goal reached:", reached)

evaluated policy
path: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]
steps: 8
total reward: 3
goal reached: True


In [14]:
# optimal policy
def optimal_policy_func(state):
    return optimal_policy[state]

path, steps, total_reward, reached = run_episode(optimal_policy_func)
print("optimal policy")
print("path:", path)
print("steps:", steps)
print("total reward:", total_reward)
print("goal reached:", reached)

optimal policy
path: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]
steps: 8
total reward: 3
goal reached: True
